# Прогноз золото-урановых зон: Random Forest

Этот ноутбук строит сетку 500×500 м, рассчитывает геологические признаки по расстояниям до факторов, обучает **Random Forest** по точкам проявлений/аномалий и отображает карту перспективности.

**Идея:** не просто вручную складывать факторы, а использовать ML-модель, которая учится на известных точках.

In [1]:
# При необходимости раскомментируй установку библиотек
# !pip install geopandas shapely scikit-learn matplotlib pandas numpy pyogrio

import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely.geometry import box
from shapely.ops import unary_union

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)


## 1. Настройки путей

Поменяй `BASE_DIR` на свою папку проекта. Внутри должна быть папка `shp_dbf` с shp-файлами.

In [2]:
# === ГЛАВНАЯ НАСТРОЙКА ===
BASE_DIR = r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"
SHP_DIR = os.path.join(BASE_DIR, "shp_dbf")
OUT_DIR = os.path.join(BASE_DIR, "rf_forecast_result")
os.makedirs(OUT_DIR, exist_ok=True)

# Размер ячейки как в методичке: 500 x 500 метров
CELL_SIZE = 500

print("BASE_DIR:", BASE_DIR)
print("SHP_DIR:", SHP_DIR)
print("OUT_DIR:", OUT_DIR)


BASE_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз
SHP_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\shp_dbf
OUT_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\rf_forecast_result


## 2. Вспомогательные функции

In [3]:
def find_shp_files(folder):
    """Список всех shapefile в папке."""
    return sorted([p for p in Path(folder).glob("*.shp")])


def load_layer(path, fallback_crs="EPSG:4326"):
    """Загрузка слоя + защита от отсутствующего CRS."""
    gdf = gpd.read_file(path)
    gdf = gdf[~gdf.geometry.isna()].copy()
    if gdf.crs is None:
        print(f"CRS не найден → задаем {fallback_crs} для {Path(path).name}")
        gdf = gdf.set_crs(fallback_crs)
    return gdf


def to_metric_crs(gdf):
    """Перевод в метрическую проекцию. Если слой уже в метрах — оставляем."""
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    try:
        epsg = gdf.crs.to_epsg()
    except Exception:
        epsg = None
    
    # Если CRS географический, переводим в Web Mercator для метровых расстояний
    if gdf.crs.is_geographic:
        return gdf.to_crs("EPSG:3857")
    return gdf


def normalize_01(values):
    """Нормализация массива в диапазон 0..1."""
    arr = np.asarray(values, dtype="float64")
    mn = np.nanmin(arr)
    mx = np.nanmax(arr)
    if not np.isfinite(mn) or not np.isfinite(mx) or mx - mn == 0:
        return np.zeros_like(arr)
    return (arr - mn) / (mx - mn)


def distance_to_union(points_gdf, target_gdf):
    """Расстояние от центров ячеек до объединенной геометрии слоя."""
    if target_gdf.empty:
        return np.full(len(points_gdf), np.nan)
    geom_union = unary_union(target_gdf.geometry.values)
    return points_gdf.geometry.distance(geom_union).values


def proximity_from_distance(dist, scale):
    """Перевод расстояния в близость: 1 — рядом, 0 — далеко."""
    dist = np.asarray(dist, dtype="float64")
    dist = np.where(np.isfinite(dist), dist, np.nanmax(dist[np.isfinite(dist)]) if np.any(np.isfinite(dist)) else 0)
    return 1.0 / (1.0 + dist / scale)


## 3. Загрузка слоев

Основные слои берутся по именам файлов. Если у тебя названия отличаются, измени словарь `LAYER_FILES`.

In [4]:
all_shp = find_shp_files(SHP_DIR)
print("Найденные shp-файлы:")
for p in all_shp:
    print(" -", p.name)

# Слои факторов
LAYER_FILES = {
    "mask": "svita_new.shp",          # область прогноза / свиты
    "facies": "fasii.shp",           # литолого-фациальный фактор
    "paleo": "gr_dol_vp_poly.shp",   # палеодолины и впадины
    "struct": "kory.shp",            # коры выветривания
    "magm": "dayki_buf.shp",         # дайки
    "tect1": "glub_raz_nw.shp",      # глубинные разломы
    "tect2": "glub_r_nw.shp",        # региональные разломы
}

layers = {}
for key, fname in LAYER_FILES.items():
    path = os.path.join(SHP_DIR, fname)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Не найден файл: {path}")
    layers[key] = load_layer(path)
    print(f"{key:>7}: {fname:20s} | объектов: {len(layers[key])} | CRS: {layers[key].crs}")


Найденные shp-файлы:
 - dayki_buf.shp
 - fasii.shp
 - glub_r_nw.shp
 - glub_raz_nw.shp
 - gr_dol_vp_poly.shp
 - kory.shp
 - result.shp
 - svita_new.shp
 - геохимические ореолы.shp
 - геохимическое_опробование.shp
 - привнос урана.shp
CRS не найден → задаем EPSG:4326 для svita_new.shp
   mask: svita_new.shp        | объектов: 101 | CRS: EPSG:4326
CRS не найден → задаем EPSG:4326 для fasii.shp
 facies: fasii.shp            | объектов: 15 | CRS: EPSG:4326
CRS не найден → задаем EPSG:4326 для gr_dol_vp_poly.shp
  paleo: gr_dol_vp_poly.shp   | объектов: 6 | CRS: EPSG:4326
CRS не найден → задаем EPSG:4326 для kory.shp
 struct: kory.shp             | объектов: 19 | CRS: EPSG:4326
CRS не найден → задаем EPSG:4326 для dayki_buf.shp
   magm: dayki_buf.shp        | объектов: 151 | CRS: EPSG:4326
CRS не найден → задаем EPSG:4326 для glub_raz_nw.shp
  tect1: glub_raz_nw.shp      | объектов: 4 | CRS: EPSG:4326
CRS не найден → задаем EPSG:4326 для glub_r_nw.shp
  tect2: glub_r_nw.shp        | объекто

## 4. Приведение CRS и построение сетки 500×500 м

In [6]:
# Приводим маску к метрической CRS
mask = to_metric_crs(layers["mask"])
target_crs = mask.crs

# Все факторы приводим к CRS маски
for key in layers:
    layers[key] = to_metric_crs(layers[key]).to_crs(target_crs)

mask_union = unary_union(mask.geometry.values)
minx, miny, maxx, maxy = mask.total_bounds

cells = []
cell_ids = []
row_ids = []
col_ids = []

idx = 0
x_values = np.arange(minx, maxx, CELL_SIZE)
y_values = np.arange(miny, maxy, CELL_SIZE)

for col, x in enumerate(x_values):
    for row, y in enumerate(y_values):
        geom = box(x, y, x + CELL_SIZE, y + CELL_SIZE)
        # берем ячейки, которые пересекают маску
        if geom.intersects(mask_union):
            cells.append(geom)
            cell_ids.append(idx)
            row_ids.append(row)
            col_ids.append(col)
            idx += 1

grid = gpd.GeoDataFrame(
    {"cell_id": cell_ids, "grid_row": row_ids, "grid_col": col_ids},
    geometry=cells,
    crs=target_crs,
)

# Обрезаем ячейки по маске визуально/геометрически
# Для скорости оставляем квадратные ячейки, но фильтр уже сделан через intersects
print("Количество ячеек:", len(grid))
grid.head()


ValueError: arange: cannot compute length

## 5. Расчет признаков для Random Forest

Для каждого фактора считаем расстояние от центра ячейки до объекта, затем переводим расстояние в близость `prox_*`.

Чем ближе объект, тем значение ближе к 1.

In [ ]:
centers = grid.copy()
centers["geometry"] = centers.geometry.centroid

# Масштабы влияния факторов в метрах
SCALES = {
    "facies": 1000,
    "paleo": 1000,
    "struct": 900,
    "magm": 800,
    "tect1": 800,
    "tect2": 800,
}

for key, scale in SCALES.items():
    dist = distance_to_union(centers, layers[key])
    grid[f"dist_{key}"] = dist
    grid[f"prox_{key}"] = proximity_from_distance(dist, scale)

# Дополнительные признаки-взаимодействия
prox_tect_mean = (grid["prox_tect1"] + grid["prox_tect2"]) / 2

grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["tect_magm_intersection"] = prox_tect_mean * grid["prox_magm"]
grid["tect_struct_intersection"] = prox_tect_mean * grid["prox_struct"]
grid["paleo_struct_intersection"] = grid["prox_paleo"] * grid["prox_struct"]

# Экспертный скор оставляем только как дополнительный признак, не как итоговый ответ
geo_raw = (
    1.4 * grid["prox_tect1"] +
    1.4 * grid["prox_tect2"] +
    1.2 * grid["prox_magm"] +
    1.1 * grid["prox_struct"] +
    1.0 * grid["prox_paleo"] +
    0.9 * grid["prox_facies"] +
    0.5 * grid["tect_intersection"] +
    0.4 * grid["tect_magm_intersection"] +
    0.3 * grid["tect_struct_intersection"] +
    0.2 * grid["paleo_struct_intersection"]
)
grid["geo_score"] = normalize_01(geo_raw)

feature_cols = [
    "prox_facies", "prox_paleo", "prox_struct", "prox_magm", "prox_tect1", "prox_tect2",
    "tect_intersection", "tect_magm_intersection", "tect_struct_intersection", "paleo_struct_intersection",
    "geo_score",
]

grid[feature_cols].describe()


## 6. Поиск точек минерализации / аномалий

Ноутбук автоматически ищет точечные shp-слои. Если нужные точки не найдутся, укажи файл вручную в `POINT_LAYER_NAME`.

In [ ]:
# Автоматический поиск точечных слоев
point_candidates = []
for p in all_shp:
    try:
        gdf = load_layer(p)
        geom_types = sorted(gdf.geometry.geom_type.unique().tolist())
        if any(t in ["Point", "MultiPoint"] for t in geom_types):
            point_candidates.append({"file": p.name, "path": str(p), "n": len(gdf), "geom_types": geom_types})
    except Exception as e:
        pass

point_candidates_df = pd.DataFrame(point_candidates)
point_candidates_df


In [ ]:
# Если автоматический выбор не подходит, напиши имя файла вручную, например: "result.shp"
POINT_LAYER_NAME = None

if POINT_LAYER_NAME is not None:
    point_path = os.path.join(SHP_DIR, POINT_LAYER_NAME)
elif len(point_candidates) > 0:
    # Берем самый крупный точечный слой как основной источник положительных точек
    point_path = max(point_candidates, key=lambda d: d["n"])["path"]
else:
    point_path = None

if point_path is None:
    raise RuntimeError("Точечный слой не найден. Укажи POINT_LAYER_NAME вручную.")

points = load_layer(point_path)
points = to_metric_crs(points).to_crs(target_crs)
print("Используем точечный слой:", Path(point_path).name)
print("Количество точек:", len(points))
points.head()


## 7. Создание целевой переменной `target`

`target = 1`, если в ячейку попадает точка проявления/аномалии. Остальные ячейки получают `0`.

In [ ]:
# Оставляем только точки внутри области прогноза
points_in_mask = points[points.geometry.within(mask_union)].copy()
print("Точек внутри маски:", len(points_in_mask))

# spatial join: точка попала в ячейку
join = gpd.sjoin(
    points_in_mask[["geometry"]],
    grid[["cell_id", "geometry"]],
    how="inner",
    predicate="within",
)

positive_cell_ids = set(join["cell_id"].unique().tolist())
grid["target"] = grid["cell_id"].isin(positive_cell_ids).astype(int)

print("Positive cells:", int(grid["target"].sum()))
print("Total cells:", len(grid))
print("Base rate:", grid["target"].mean())

if grid["target"].sum() < 5:
    print("ВНИМАНИЕ: положительных ячеек очень мало. Метрики могут быть нестабильными.")


## 8. Обучение Random Forest

In [ ]:
X = grid[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = grid["target"].astype(int)

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=8,
    min_samples_leaf=5,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)

rf.fit(X, y)
grid["rf_prospectivity"] = rf.predict_proba(X)[:, 1]

# Для совместимости с методичкой: меньше prognoz → перспективнее
# Но для карты удобнее использовать rf_prospectivity: больше → перспективнее
grid["prognoz"] = 1 - grid["rf_prospectivity"]

print("Обучение завершено")
print("min/max rf_prospectivity:", grid["rf_prospectivity"].min(), grid["rf_prospectivity"].max())


## 9. Метрики качества

Важно: метрики на обучающей выборке могут быть завышены. Ниже добавлена грубая пространственная валидация по группам сетки.

In [ ]:
# Метрики на всей обучающей выборке
proba = grid["rf_prospectivity"].values
pred05 = (proba >= 0.5).astype(int)

metrics = {}
if y.nunique() == 2 and y.sum() > 0:
    metrics["ROC_AUC_train"] = float(roc_auc_score(y, proba))
    metrics["Average_Precision_train"] = float(average_precision_score(y, proba))
    metrics["Precision_0_5_train"] = float(precision_score(y, pred05, zero_division=0))
    metrics["Recall_0_5_train"] = float(recall_score(y, pred05, zero_division=0))
    metrics["F1_0_5_train"] = float(f1_score(y, pred05, zero_division=0))
    metrics["Confusion_Matrix_train"] = confusion_matrix(y, pred05).tolist()
else:
    print("Недостаточно классов для расчета метрик")

pd.DataFrame([metrics])


In [ ]:
# Пространственная валидация: делим сетку на крупные блоки
# Это честнее, чем случайный train/test split, потому что соседние ячейки похожи.

grid["spatial_group"] = (grid["grid_col"] // 10).astype(str) + "_" + (grid["grid_row"] // 10).astype(str)
groups = grid["spatial_group"].values

unique_groups = np.unique(groups)
n_splits = min(5, len(unique_groups))

cv_results = []
if y.nunique() == 2 and y.sum() >= n_splits and n_splits >= 2:
    gkf = GroupKFold(n_splits=n_splits)
    fold = 1
    for train_idx, test_idx in gkf.split(X, y, groups):
        model = RandomForestClassifier(
            n_estimators=500,
            max_depth=8,
            min_samples_leaf=5,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1,
        )
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        p = model.predict_proba(X.iloc[test_idx])[:, 1]
        yy = y.iloc[test_idx]
        
        row = {"fold": fold, "positive_test": int(yy.sum()), "n_test": len(yy)}
        if yy.nunique() == 2:
            row["ROC_AUC"] = roc_auc_score(yy, p)
            row["Average_Precision"] = average_precision_score(yy, p)
        else:
            row["ROC_AUC"] = np.nan
            row["Average_Precision"] = np.nan
        cv_results.append(row)
        fold += 1

cv_df = pd.DataFrame(cv_results)
cv_df


## 10. Важность признаков

In [ ]:
importances = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False)

importances


In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(importances["feature"][::-1], importances["importance"][::-1])
plt.title("Random Forest: важность признаков")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 11. Классы перспективности

In [ ]:
# Делим на 5 классов по квантилям
labels = ["very_low", "low", "medium", "high", "very_high"]
grid["prospectivity_class"] = pd.qcut(
    grid["rf_prospectivity"].rank(method="first"),
    q=5,
    labels=labels,
)

# Верхние 10% как наиболее перспективные зоны
top10_threshold = grid["rf_prospectivity"].quantile(0.90)
grid["top10_zone"] = (grid["rf_prospectivity"] >= top10_threshold).astype(int)

# HitRate в Top10
if grid["target"].sum() > 0:
    hitrate_top10 = grid.loc[grid["top10_zone"] == 1, "target"].sum() / grid["target"].sum()
    precision_top10 = grid.loc[grid["top10_zone"] == 1, "target"].mean()
    base_rate = grid["target"].mean()
    lift_top10 = precision_top10 / base_rate if base_rate > 0 else np.nan
    print("HitRate Top10:", hitrate_top10)
    print("Precision Top10:", precision_top10)
    print("Lift Top10:", lift_top10)


## 12. Отображение результата

На карте: чем выше `rf_prospectivity`, тем перспективнее территория.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))

grid.plot(
    column="rf_prospectivity",
    ax=ax,
    cmap="YlOrRd",
    legend=True,
    linewidth=0,
    legend_kwds={"label": "RF prospectivity: выше = перспективнее"},
)

# граница маски
mask.boundary.plot(ax=ax, color="black", linewidth=0.8)

# точки проявлений
if len(points_in_mask) > 0:
    points_in_mask.plot(ax=ax, color="cyan", edgecolor="black", markersize=25, label="Known points")

ax.set_title("Прогноз перспективности: Random Forest", fontsize=14)
ax.set_axis_off()
ax.legend(loc="lower right")
plt.tight_layout()

png_path = os.path.join(OUT_DIR, "rf_prospectivity_map.png")
plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.show()

print("PNG сохранен:", png_path)


In [ ]:
# Карта top-10% зон
fig, ax = plt.subplots(figsize=(10, 12))

grid.plot(
    column="top10_zone",
    ax=ax,
    cmap="Set1",
    legend=True,
    linewidth=0,
)
mask.boundary.plot(ax=ax, color="black", linewidth=0.8)
if len(points_in_mask) > 0:
    points_in_mask.plot(ax=ax, color="yellow", edgecolor="black", markersize=25, label="Known points")

ax.set_title("Top-10% наиболее перспективных зон по Random Forest", fontsize=14)
ax.set_axis_off()
ax.legend(loc="lower right")
plt.tight_layout()

top_png_path = os.path.join(OUT_DIR, "rf_top10_zones.png")
plt.savefig(top_png_path, dpi=300, bbox_inches="tight")
plt.show()

print("PNG сохранен:", top_png_path)


## 13. Сохранение результатов

In [ ]:
gpkg_path = os.path.join(OUT_DIR, "rf_forecast_result.gpkg")
csv_path = os.path.join(OUT_DIR, "rf_grid_attributes.csv")
metrics_path = os.path.join(OUT_DIR, "rf_metrics.json")

# Сохраняем карту в GPKG
grid.to_file(gpkg_path, layer="rf_grid", driver="GPKG")

# Сохраняем таблицу без геометрии
attr = grid.drop(columns="geometry")
attr.to_csv(csv_path, index=False, encoding="utf-8-sig")

# Метрики
save_metrics = metrics.copy()
if "cv_df" in globals() and len(cv_df) > 0:
    save_metrics["CV_ROC_AUC_mean"] = float(cv_df["ROC_AUC"].mean(skipna=True))
    save_metrics["CV_AP_mean"] = float(cv_df["Average_Precision"].mean(skipna=True))

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(save_metrics, f, ensure_ascii=False, indent=2)

print("GPKG:", gpkg_path)
print("CSV:", csv_path)
print("Metrics:", metrics_path)


## 14. Что писать в презентации

**Random Forest** был выбран как нелинейная ML-модель, способная учитывать совместное влияние геологических факторов.  
В отличие от простого `geo_score`, модель обучается на известных точках проявлений и сама оценивает вклад признаков.

**Ограничение:** данных с положительными примерами мало, поэтому качество нужно оценивать осторожно, особенно с учетом пространственной валидации.